In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Configuration ---
# Update these paths if your output directory is named differently
CKPT_8K_PATH = "./gemma3-141M-kinyarwanda-cpt-test-1/checkpoint-8000"
CKPT_10K_PATH = "./gemma3-141M-kinyarwanda-cpt-test-1/checkpoint-10000"




# We only need to load the tokenizer once, as both checkpoints share it
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CKPT_10K_PATH, use_fast=True)

# Load both models into VRAM using bfloat16 for speed
print("Loading Checkpoint 8000...")
model_8k = AutoModelForCausalLM.from_pretrained(
    CKPT_8K_PATH, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
).eval()

print("Loading Checkpoint 10000...")
model_10k = AutoModelForCausalLM.from_pretrained(
    CKPT_10K_PATH, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
).eval()

# --- Define Your Test Prompts ---
prompts = [
    "Uyu munsi ni umunsi mwiza kuko",         # Kinyarwanda continuation
    "Mu Rwanda, amakuru y'uyu munsi ni",       # News context continuation
    "In Kinyarwanda, the word for 'water' is"  # Cross-lingual mapping attempt
]


prompts = [
    "Uyu munsi ni umunsi mwiza kuko",  # Fluency check
    "Amakuru yawe? Mfite",              # Conversational check
     "Imana", 
     "mugabo yaraje abwira abantu ati"
     "mu gihugu cy'ubufaransa",
     "amateka ya Afurika", 
     "Ejo bundi umugabo yaje nk'iya Gatera",
      "umwana wange yarambwiye",
      "Ejo bundi umwana yagiye"  , 
      "Umugore yatawe muri yombi akekwaho", 
    "The capital of Rwanda is Kigali. In Kinyarwanda, this translates to",  # Cross-lingual hint
      "the history of the persian empire"    
]



# --- Helper Function for Generation ---
def generate_text(model, prompt, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=40,       # Adjust length as needed
            do_sample=True,          # Allow some natural variance
            temperature=0.6,         # Keep it focused but not completely rigid
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Run the Comparison ---
print("\n" + "="*50)
print("🏁 STARTING GENERATION COMPARISON 🏁")
print("="*50)

for idx, prompt in enumerate(prompts):
    print(f"\n[Prompt {idx+1}]: {prompt}")
    print("-" * 50)
    
    # Generate with 8K
    out_8k = generate_text(model_8k, prompt, tokenizer)
    print(f"🔹 Checkpoint 8000:\n{out_8k}\n")
    
    # Generate with 10K
    out_10k = generate_text(model_10k, prompt, tokenizer)
    print(f"🔸 Checkpoint 10000:\n{out_10k}")
    print("=" * 50)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Tokenizer...
Loading Checkpoint 8000...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loading Checkpoint 10000...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]


🏁 STARTING GENERATION COMPARISON 🏁

[Prompt 1]: Uyu munsi ni umunsi mwiza kuko
--------------------------------------------------
🔹 Checkpoint 8000:
 Uyu munsi ni umunsi mwiza kuko ushobora no gutanga serivisi zumwuga. “I am a very creative and passionate individual. I have a passion for design and want to make a difference in the world. I love to think creatively

🔸 Checkpoint 10000:
 Uyu munsi ni umunsi mwiza kuko uba ufite n'uburyo bwiza bwo kwidagadura.

[Prompt 2]: Amakuru yawe? Mfite
--------------------------------------------------
🔹 Checkpoint 8000:
 Amakuru yawe? Mfite uburambe bwimyaka irenga icumi mugukora no kugurisha ibicuruzwa. Muri make, ibyuma bidafite ibyuma, ibyuma bidafite ibyuma, ibyuma bidafite ibyuma, ibyuma bidafite ibyuma, ibyuma bidafite ibyuma, ibyuma bidafite ibyuma, ibyuma

🔸 Checkpoint 10000:
 Amakuru yawe? Mfite gahunda yumunsi w'abakundana. Nyuma yiminsi mike, abakiriya bacu benshi, ibigo byinshi, namasosiyete yo hanze bafite igihe cyo gutanga. The best

In [3]:

prompts = [
    "Uyu munsi ni umunsi mwiza kuko",  # Fluency check
    "Amakuru yawe? Mfite",              # Conversational check
     "Imana ", 
     "mugabo yaraje abwira abantu ati "
     "mu gihugu cy'ubufaransa",
     "amateka ya Afurika", 
     "Ejo bundi umugabo yaje nk'iya Gatera",
      "umwana wange yarambwiye   ",
      "Ejo bundi umwana yagiye "  , 
      "Umugore yatawe muri yombi akekwaho", 
    "The capital of Rwanda is Kigali. In Kinyarwanda, this translates to",  # Cross-lingual hint
      "the history of the persian empire "    
]



# --- Helper Function for Generation ---
def generate_text(model, prompt, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=40,       # Adjust length as needed
            do_sample=True,          # Allow some natural variance
            temperature=0.6,         # Keep it focused but not completely rigid
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Run the Comparison ---
print("\n" + "="*50)
print("🏁 STARTING GENERATION COMPARISON 🏁")
print("="*50)

for idx, prompt in enumerate(prompts):
    print(f"\n[Prompt {idx+1}]: {prompt}")
    print("-" * 50)
    
    # Generate with 8K
    out_8k = generate_text(model_8k, prompt, tokenizer)
    print(f"🔹 Checkpoint 8000:\n{out_8k}\n")
    
    # Generate with 10K
    out_10k = generate_text(model_10k, prompt, tokenizer)
    print(f"🔸 Checkpoint 10000:\n{out_10k}")
    print("=" * 50)


🏁 STARTING GENERATION COMPARISON 🏁

[Prompt 1]: Uyu munsi ni umunsi mwiza kuko
--------------------------------------------------
🔹 Checkpoint 8000:
 Uyu munsi ni umunsi mwiza kuko ushobora gukora ibintu byose ukeneye byose mu murima. Mu minsi mikuru ya Noheri, abantu benshi bakoresha uburyo bwo gukora imitako y'amabara kugirango bahuze imitako yabo. Ariko, hari ibintu byinshi ugomba gusuzuma mugihe

🔸 Checkpoint 10000:
 Uyu munsi ni umunsi mwiza kuko aba ari bwo buri gihe Imana igira inama zo gufasha abantu. Iyo Yesu amaze kubona ibintu byose abantu bari bakeneye, arabasubiza ati “ndabahaye inama yo kugira ngo n’abantu banjye bashobore kumenya

[Prompt 2]: Amakuru yawe? Mfite
--------------------------------------------------
🔹 Checkpoint 8000:
 Amakuru yawe? Mfite ikibazo? Twiyemeje gutanga ibikoresho byiza cyane. Dufite intego yo gukora ibicuruzwa byiza cyane, kandi duharanira kuba indashyikirwa mu nganda. Twizera ko abakiriya bacu bose bakeneye ibicuruzwa byiza cyane, kandi twizer